# TLC Yellow Taxi — Spark Batch ETL

**Pipeline:**  `data/raw/*.parquet`  →  Ingest  →  Clean & Transform  →  `data/processed/` (partitioned parquet)

Mirrors the original Mage/BigQuery pipeline, replacing managed cloud ETL with PySpark.

## 1. Initial SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType, TimestampType

# 建立一个SparkSession
spark = (
    SparkSession.builder
    .appName("TLC-Batch-ETL")   # 应用名，Spark UI 里显示
    .config("spark.sql.shuffle.partitions", "8")   # shuffle 时分几个分区，本地运行调小，默认 200 太多
    .config("spark.driver.memory", "4g")   # 用多少内存
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("SparkSession ready.")

26/09/28 19:42:39 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/09/28 19:42:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/28 19:42:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1
SparkSession ready.


## 2. Ingest original Parquet

In [2]:
import os

RAW_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/raw/*.parquet")

raw_df = spark.read.parquet(RAW_PATH)

print(f"Row count: {raw_df.count():,}")
print(f"Columns : {len(raw_df.columns)}")
raw_df.printSchema()

Row count: 9,554,778
Columns : 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
# 快速预览
raw_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

## 3. Quality Check

In [4]:
from pyspark.sql.functions import col, count, when, isnan, isnull

key_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
            "trip_distance", "fare_amount", "total_amount",
            "PULocationID", "DOLocationID", "passenger_count"]

null_counts = raw_df.select([
    count(when(isnull(c), c)).alias(c) for c in key_cols
])
print("Null counts:")
null_counts.show()

# 异常值分布
raw_df.select(
    F.min("fare_amount").alias("fare_min"),
    F.max("fare_amount").alias("fare_max"),
    F.min("trip_distance").alias("dist_min"),
    F.max("trip_distance").alias("dist_max"),
    F.min("passenger_count").alias("pax_min"),
    F.max("passenger_count").alias("pax_max"),
).show()

Null counts:


+--------------------+---------------------+-------------+-----------+------------+------------+------------+---------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|PULocationID|DOLocationID|passenger_count|
+--------------------+---------------------+-------------+-----------+------------+------------+------------+---------------+
|                   0|                    0|            0|          0|           0|           0|           0|         751962|
+--------------------+---------------------+-------------+-----------+------------+------------+------------+---------------+



+--------+--------+--------+--------+-------+-------+
|fare_min|fare_max|dist_min|dist_max|pax_min|pax_max|
+--------+--------+--------+--------+-------+-------+
|  -999.0|  9792.0|     0.0|312722.3|      0|      9|
+--------+--------+--------+--------+-------+-------+



## 4. Transform — 清洗 + 衍生列

In [5]:
from pyspark.sql.functions import (
    col, unix_timestamp, round as spark_round,
    hour, dayofweek, month, year, lit, when
)

# ── 4.1 过滤无效记录 ──────────────────────────────────────────────────────
cleaned = (
    raw_df
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    .filter(col("passenger_count").between(1, 6))
    .filter(col("tpep_pickup_datetime") >= "2024-01-01")  # 过滤错误年份（2002/2008/2009）
    .filter(col("tpep_pickup_datetime") <  "2024-04-01")  # 过滤混入的 4 月数据
)

removed = raw_df.count() - cleaned.count()
print(f"Removed {removed:,} invalid rows  ({removed/raw_df.count()*100:.1f}%)")

26/09/28 19:42:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Removed 1,075,368 invalid rows  (11.3%)


In [6]:
# ── 4.2 添加衍生列 ────────────────────────────────────────────────────────
transformed = (
    cleaned
    # 行程时长（分钟）
    .withColumn(
        "trip_duration_min",
        spark_round(
            (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
            2
        )
    )
    # 平均时速（mph）
    .withColumn(
        "avg_speed_mph",
        spark_round(col("trip_distance") / (col("trip_duration_min") / 60), 2)
    )
    # 每英里费用
    .withColumn(
        "fare_per_mile",
        spark_round(col("fare_amount") / col("trip_distance"), 2)
    )
    # 时段（对应 Looker 里的 time-of-day 分析）
    .withColumn(
        "time_of_day",
        when(hour("tpep_pickup_datetime").between(6, 9), "morning_rush")
        .when(hour("tpep_pickup_datetime").between(10, 15), "midday")
        .when(hour("tpep_pickup_datetime").between(16, 19), "evening_rush")
        .when(hour("tpep_pickup_datetime").between(20, 23), "night")
        .otherwise("late_night")
    )
    # 日期维度列（用于分区 + 分析）
    .withColumn("pickup_year",  year("tpep_pickup_datetime"))
    .withColumn("pickup_month", month("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  hour("tpep_pickup_datetime"))
    .withColumn("day_of_week",  dayofweek("tpep_pickup_datetime"))  # 1=Sun, 7=Sat
    # 支付方式映射（1=Credit, 2=Cash, 3=No charge, 4=Dispute）
    .withColumn(
        "payment_label",
        when(col("payment_type") == 1, "credit_card")
        .when(col("payment_type") == 2, "cash")
        .when(col("payment_type") == 3, "no_charge")
        .otherwise("other")
    )
    # 过滤明显异常时速（> 100 mph 或负数）
    .filter(col("avg_speed_mph").between(0, 100))
    .filter(col("trip_duration_min").between(1, 180))
)

print(f"Final row count: {transformed.count():,}")
transformed.select(
    "tpep_pickup_datetime", "trip_duration_min", "avg_speed_mph",
    "fare_per_mile", "time_of_day", "payment_label"
).show(5)

Final row count: 8,447,443


+--------------------+-----------------+-------------+-------------+-----------+-------------+
|tpep_pickup_datetime|trip_duration_min|avg_speed_mph|fare_per_mile|time_of_day|payment_label|
+--------------------+-----------------+-------------+-------------+-----------+-------------+
| 2024-01-01 00:57:55|             19.8|         5.21|        10.29| late_night|         cash|
| 2024-01-01 00:03:00|              6.6|        16.36|         5.56| late_night|  credit_card|
| 2024-01-01 00:17:06|            17.92|        15.74|         4.96| late_night|  credit_card|
| 2024-01-01 00:36:38|              8.3|        10.12|         7.14| late_night|  credit_card|
| 2024-01-01 00:46:51|              6.1|         7.87|         9.88| late_night|  credit_card|
+--------------------+-----------------+-------------+-------------+-----------+-------------+
only showing top 5 rows



## 5. Load — 写出分区 Parquet

In [ ]:
%%time
OUTPUT_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/processed/trips")

(
    transformed
    .repartition("pickup_year", "pickup_month")   # 同一月的数据集中到同一任务，每月输出 1 个文件，避免小文件
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

print(f"Written to: {OUTPUT_PATH}")

# 验证分区目录
import subprocess
result = subprocess.run(["find", OUTPUT_PATH, "-name", "*.parquet", "-type", "f"],
                       capture_output=True, text=True)
files = result.stdout.strip().split("\n")
print(f"Output files: {len(files)}")
for f in sorted(files):
    print(" ", f)

In [8]:
from pyspark.sql.functions import col, count

transformed.groupBy("pickup_month") \
    .agg(count("*").alias("row_count")) \
    .orderBy("pickup_month") \
    .show()

from pyspark.sql.functions import spark_partition_id

transformed.groupBy(spark_partition_id().alias("partition_id"), "pickup_month") \
    .agg(count("*").alias("row_count")) \
    .orderBy("partition_id", "pickup_month") \
    .show(30)


+------------+---------+
|pickup_month|row_count|
+------------+---------+
|           1|  2713403|
|           2|  2709721|
|           3|  3024319|
+------------+---------+



+------------+------------+---------+
|partition_id|pickup_month|row_count|
+------------+------------+---------+
|           0|           1|  1005835|
|           1|           1|  1707558|
|           1|           2|        3|
|           2|           1|       10|
|           2|           2|  1007622|
|           3|           2|  1702077|
|           3|           3|        2|
|           4|           2|       19|
|           4|           3|  1005459|
|           5|           3|  1005195|
|           6|           3|  1013663|
+------------+------------+---------+



## 6. 指标汇总 — 模拟 Looker Dashboard 的核心聚合

In [9]:
# 从分区输出重新读（验证 write 正确）
processed_df = spark.read.parquet(OUTPUT_PATH)

# ── 6.1 时段 × 每日需求量 ─────────────────────────────────────────────────
print("=== Trip Volume by Time of Day ===")
(
    processed_df
    .groupBy("pickup_month", "time_of_day")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min")
    )
    .orderBy("pickup_month", "time_of_day")
).show(20)

=== Trip Volume by Time of Day ===


+------------+------------+----------+--------+----------------+
|pickup_month| time_of_day|trip_count|avg_fare|avg_duration_min|
+------------+------------+----------+--------+----------------+
|           1|evening_rush|    733633|   17.96|            15.2|
|           1|  late_night|    194153|   19.21|            13.2|
|           1|      midday|    928851|   18.36|            15.8|
|           1|morning_rush|    333192|   18.33|            15.1|
|           1|       night|    523574|   18.81|            13.7|
|           2|evening_rush|    737649|   18.02|            15.8|
|           2|  late_night|    195197|   18.04|            12.7|
|           2|      midday|    887926|   18.76|            16.5|
|           2|morning_rush|    332446|    18.0|            15.0|
|           2|       night|    556503|    18.4|            14.0|
|           3|evening_rush|    803809|   18.66|            16.4|
|           3|  late_night|    228725|   19.26|            13.5|
|           3|      midda

In [10]:
# ── 6.2 Top 10 热门上车区域（对应 pickup/dropoff patterns）────────────────
print("=== Top 10 Pickup Zones ===")
(
    processed_df
    .groupBy("PULocationID")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.desc("trip_count"))
    .limit(10)
).show()

=== Top 10 Pickup Zones ===


+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         161|    417364|
|         237|    409651|
|         132|    400196|
|         236|    381519|
|         162|    312827|
|         230|    300701|
|         186|    299711|
|         142|    289088|
|         138|    274017|
|         163|    254194|
+------------+----------+



In [11]:
# ── 6.3 支付方式分布 ──────────────────────────────────────────────────────
print("=== Payment Method Distribution ===")
(
    processed_df
    .groupBy("payment_label")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy(F.desc("trip_count"))
).show()

=== Payment Method Distribution ===


+-------------+----------+-------+
|payment_label|trip_count|avg_tip|
+-------------+----------+-------+
|  credit_card|   7097556|   4.19|
|         cash|   1250471|    0.0|
|        other|     70304|   0.01|
|    no_charge|     29112|   0.01|
+-------------+----------+-------+



In [12]:
spark.stop()
print("Spark stopped.")

Spark stopped.
